In [1]:
import math

# --- 基础数学工具函数 ---

def norm_cdf(x):
    """标准正态分布累积分布函数 (Cumulative Distribution Function)"""
    return (1.0 + math.erf(x / math.sqrt(2.0))) / 2.0

def norm_pdf(x):
    """标准正态分布概率密度函数 (Probability Density Function)"""
    return math.exp(-0.5 * x**2) / math.sqrt(2.0 * math.pi)

# --- Black-Scholes 核心函数 ---

def bs_call_price(S, K, T, r, q, sigma):
    """计算欧式看涨期权价格"""
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    price = S * math.exp(-q * T) * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)
    return price

def bs_call_delta(S, K, T, r, q, sigma):
    """计算欧式看涨期权 Delta"""
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    return math.exp(-q * T) * norm_cdf(d1)

def bs_vega(S, K, T, r, q, sigma):
    """计算 Vega (用于反推隐含波动率)"""
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    return S * math.exp(-q * T) * norm_pdf(d1) * math.sqrt(T)

def implied_volatility(market_price, S, K, T, r, q):
    """
    使用牛顿-拉夫逊法 (Newton-Raphson) 计算隐含波动率
    """
    sigma = 0.5  # 初始猜测值
    tol = 1e-8   # 容忍误差
    max_iter = 100

    for i in range(max_iter):
        price = bs_call_price(S, K, T, r, q, sigma)
        vega = bs_vega(S, K, T, r, q, sigma)
        
        diff = market_price - price
        
        if abs(diff) < tol:
            return sigma
            
        if abs(vega) < 1e-8: # 防止除以零
            break
            
        sigma = sigma + diff / vega # 更新 sigma
        
    return sigma


In [2]:

# --- 主程序验证 ---

def main():
    print("=== Black-Scholes 计算验证 ===")
    
    # 1. 题目给定参数
    S = 55.0        # 标的价格
    r = 0.04        # 无风险利率
    q = 0.03        # 股息率
    T = 1.0         # 到期时间
    f_price = 10.94 # 衍生品市场价格
    
    # 2. 组合分解
    # Payoff = max(max(S-50, 0), 2(S-75)) 等价于 Call(50) + Call(100)
    print(f"\n参数设定:")
    print(f"S={S}, r={r}, q={q}, T={T}")
    print(f"衍生品价格 f = {f_price}")
    print("-" * 30)

    # ---------------- Part (a) ----------------
    print("\n--- Part (a): 隐含波动率分析 ---")
    
    # 计算 C(50) 的价格 (已知 vol = 35%)
    sigma_50 = 0.35
    c50_price = bs_call_price(S, 50, T, r, q, sigma_50)
    print(f"1. 计算 Call(K=50) 价格 (Vol=35%):")
    print(f"   结果 = {c50_price:.4f}")
    
    # 反推 C(100) 的市场隐含价格
    # f = C(50) + C(100) => C(100) = f - C(50)
    c100_market_price = f_price - c50_price
    print(f"2. 反推 Call(K=100) 的市场隐含价格:")
    print(f"   {f_price} - {c50_price:.4f} = {c100_market_price:.4f}")
    
    # 计算 C(100) 的隐含波动率
    sigma_100 = implied_volatility(c100_market_price, S, 100, T, r, q)
    print(f"3. 计算 Call(K=100) 的隐含波动率:")
    print(f"   结果 = {sigma_100:.4%}")
    
    if sigma_100 > sigma_50:
        print("   结论: Vol(100) > Vol(50)，存在 Volatility Smile/Skew (波动率偏斜)。")
    else:
        print("   结论: 波动率持平或下降。")

    # ---------------- Part (b) ----------------
    print("\n--- Part (b): Delta 计算 ---")
    
    # 计算 Delta 50 (使用 vol = 35%)
    delta_50 = bs_call_delta(S, 50, T, r, q, sigma_50)
    print(f"1. Delta (K=50, Vol=35%):")
    print(f"   结果 = {delta_50:.4f}")
    
    # 计算 Delta 100 (使用 vol = 40.30%)
    delta_100 = bs_call_delta(S, 100, T, r, q, sigma_100)
    print(f"2. Delta (K=100, Vol={sigma_100:.2%}):")
    print(f"   结果 = {delta_100:.4f}")
    
    # 总 Delta
    total_delta = delta_50 + delta_100
    print(f"3. 总 Delta (f):")
    print(f"   {delta_50:.4f} + {delta_100:.4f} = {total_delta:.4f}")

if __name__ == "__main__":
    main()

=== Black-Scholes 计算验证 ===

参数设定:
S=55.0, r=0.04, q=0.03, T=1.0
衍生品价格 f = 10.94
------------------------------

--- Part (a): 隐含波动率分析 ---
1. 计算 Call(K=50) 价格 (Vol=35%):
   结果 = 10.0247
2. 反推 Call(K=100) 的市场隐含价格:
   10.94 - 10.0247 = 0.9153
3. 计算 Call(K=100) 的隐含波动率:
   结果 = 40.3001%
   结论: Vol(100) > Vol(50)，存在 Volatility Smile/Skew (波动率偏斜)。

--- Part (b): Delta 计算 ---
1. Delta (K=50, Vol=35%):
   结果 = 0.6627
2. Delta (K=100, Vol=40.30%):
   结果 = 0.1013
3. 总 Delta (f):
   0.6627 + 0.1013 = 0.7640
